In [5]:
!pip install gensim
import pandas as pd
import nltk
nltk.download('punkt_tab')

import numpy as np

from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

import gensim
from gensim.models import Word2Vec

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [7]:
data = pd.read_csv('labeledTrainData.tsv', sep='\t')

In [8]:
data.head()

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hi..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...


In [9]:
data.tail()

,id,sentiment,review
24995,3453_3,0,It seems like more consideration has gone into...
24996,5064_1,0,I don't believe they made this film. Completel...
24997,10905_3,0,"Guy is a loser. Can't get girls, needs to buil..."
24998,10194_3,0,This 30 minute documentary Buñuel made in the ...
24999,8478_8,1,I saw this movie as a child and it broke my he...


In [13]:
%%time
def preprocessing(data):
  data['review'] = data.review.str.lower()
  data['document_sentences'] = data.review.str.split('.')
  data['tokenized_sentences'] = list(
      map(lambda sentence: list(map(nltk.word_tokenize, sentence)), data.document_sentences)
      )
  data['tokenized_sentences'] = list(
      map(lambda word: list(filter(lambda lst: lst, word)), data.tokenized_sentences)
  )

preprocessing(data)

CPU times: user 37 s, sys: 246 ms, total: 37.3 s
Wall time: 37.8 s


In [14]:
data.head()

,id,sentiment,review,document_sentences,tokenized_sentences
0,5814_8,1,with all this stuff going down at the moment w...,[with all this stuff going down at the moment ...,"[[with, all, this, stuff, going, down, at, the..."
1,2381_9,1,"\the classic war of the worlds\"" by timothy hi...","[\the classic war of the worlds\"" by timothy h...","[[\the, classic, war, of, the, worlds\, '', by..."
2,7759_3,0,the film starts with a manager (nicholas bell)...,[the film starts with a manager (nicholas bell...,"[[the, film, starts, with, a, manager, (, nich..."
3,3630_4,0,it must be assumed that those who praised this...,[it must be assumed that those who praised thi...,"[[it, must, be, assumed, that, those, who, pra..."
4,9495_8,1,superbly trashy and wondrously unpretentious 8...,[superbly trashy and wondrously unpretentious ...,"[[superbly, trashy, and, wondrously, unpretent..."


In [15]:
train, test, y_train, y_test = train_test_split(data.drop(columns='sentiment'), data['sentiment'], test_size=0.2, random_state=42)

In [16]:
voc = []
for sen in train.tokenized_sentences:
  voc.extend(sen)

print(f'''
  length vocabulary = {len(voc)}
  length train = {len(train)}
  length test = {len(test)}
  length y_train = {len(y_train)}
  length y_test = {len(y_test)}
''')


  length vocabulary = 250290
  length train = 20000
  length test = 5000
  length y_train = 20000
  length y_test = 5000         



In [17]:
%%time
num_features = 300
min_word_count = 3
num_workers = 4
context = 5
downsampling = 1e-3 # 0.001

model = Word2Vec(
    sentences=voc,
    sg=1,
    hs=0,
    workers=num_workers,
    vector_size=num_features,
    window=context,
    min_count=min_word_count,
    sample=downsampling,
    negative=5,
    epochs=6
    )

CPU times: user 6min 26s, sys: 806 ms, total: 6min 26s
Wall time: 3min 52s
